In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

if not os.path.exists('razor-pay-assurance'):
    !git clone https://github.com/emilmariagiby/razor-pay-assurance.git

sys.path.append(os.path.abspath('razor-pay-assurance/backend'))
from app.learning.adapters.ieee_cis_adapter import IeeeCisAdapter


Cloning into 'razor-pay-assurance'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 231 (delta 43), reused 216 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (231/231), 895.14 KiB | 6.35 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [2]:
ieee_adapter = IeeeCisAdapter()
ieee_path = None

# Search Kaggle input directory dynamically so we never guess the path wrong
if os.path.exists('/kaggle/input'):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if filename == 'train_transaction.csv':
                ieee_path = os.path.join(dirname, filename)
                break
elif os.path.exists('../backend/data/external/ieee_cis/train_transaction.csv'):
    ieee_path = '../backend/data/external/ieee_cis/train_transaction.csv'

if not ieee_path:
    raise FileNotFoundError("\n\nCRITICAL: Could not find 'train_transaction.csv'!\nYou MUST click 'Add Data' on the right panel in Kaggle and add the 'IEEE-CIS Fraud Detection' dataset to this session.")

print(f"Loading IEEE-CIS data from: {ieee_path}")
all_records = ieee_adapter.ingest(ieee_path)
print(f"Total canonical ExternalBehavioralRecords loaded: {len(all_records)}")


Loading IEEE-CIS data from: /kaggle/input/datasets/lnasiri007/ieeecis-fraud-detection/train_transaction.csv
Total canonical ExternalBehavioralRecords loaded: 590540


In [3]:
X = []
y = []
feature_names = []

if all_records:
    feature_names = sorted(all_records[0].behavioral_features.keys())
    for record in all_records:
        X.append(record.to_vector())
        y.append(record.external_fraud_label)

X = np.array(X)
y = np.array(y)
print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")


Feature matrix shape: (590540, 5)
Target vector shape: (590540,)


In [4]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")


Training set: 472432 samples
Validation set: 118108 samples


In [5]:
print("Training Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("Training complete.")


Training Random Forest model...
Training complete.


In [6]:
y_pred = rf_model.predict(X_val)
y_prob = rf_model.predict_proba(X_val)[:, 1]

print("Classification Report:")
print(classification_report(y_val, y_pred, zero_division=0))

roc_auc = roc_auc_score(y_val, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.85      0.91    113975
           1       0.13      0.64      0.22      4133

    accuracy                           0.84    118108
   macro avg       0.56      0.75      0.56    118108
weighted avg       0.96      0.84      0.89    118108

ROC-AUC Score: 0.8296


In [7]:
MODEL_PATH = 'external_behavior_model.joblib'
joblib.dump(rf_model, MODEL_PATH)
print(f"Model successfully exported to: {MODEL_PATH}")
print("You can now download this file from Kaggle's output directory and place it in your local 'backend/data/models/' folder!")


Model successfully exported to: external_behavior_model.joblib
You can now download this file from Kaggle's output directory and place it in your local 'backend/data/models/' folder!


In [8]:
# ============================================================
# ASSURANCE — IEEE-CIS CHRONOLOGICAL VALIDATION
# KAGGLE-ONLY VERSION
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. FIND IEEE-CIS DATASET
# ------------------------------------------------------------

ieee_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "train_transaction.csv" in files:
        ieee_path = os.path.join(root, "train_transaction.csv")
        break

if ieee_path is None:
    raise FileNotFoundError(
        "train_transaction.csv was not found under /kaggle/input"
    )

print("=" * 70)
print("ASSURANCE IEEE-CIS CHRONOLOGICAL VALIDATION")
print("=" * 70)

print("Dataset:", ieee_path)

# ------------------------------------------------------------
# 2. LOAD ONLY REQUIRED IEEE-CIS COLUMNS
# ------------------------------------------------------------

required_columns = [
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",
    "card1",
    "card2",
    "isFraud",
]

df = pd.read_csv(
    ieee_path,
    usecols=required_columns
)

print("Raw dataset rows:", len(df))

# ------------------------------------------------------------
# 3. RECREATE THE 5-FEATURE SCHEMA
# ------------------------------------------------------------

# TransactionDT
df["ieee_dt"] = pd.to_numeric(
    df["TransactionDT"],
    errors="coerce"
)

# Transaction amount
df["ieee_amt"] = pd.to_numeric(
    df["TransactionAmt"],
    errors="coerce"
)

# ProductCD categorical encoding
# Use a deterministic mapping.
product_mapping = {
    value: index
    for index, value in enumerate(
        sorted(df["ProductCD"].dropna().unique())
    )
}

df["ieee_prod_enc"] = (
    df["ProductCD"]
    .map(product_mapping)
    .fillna(-1)
)

# card1
df["ieee_card1"] = pd.to_numeric(
    df["card1"],
    errors="coerce"
)

# card2
df["ieee_card2"] = pd.to_numeric(
    df["card2"],
    errors="coerce"
)

# Fraud label
df["label"] = pd.to_numeric(
    df["isFraud"],
    errors="coerce"
)

# ------------------------------------------------------------
# 4. REMOVE INVALID ROWS
# ------------------------------------------------------------

feature_columns = [
    "ieee_dt",
    "ieee_amt",
    "ieee_prod_enc",
    "ieee_card1",
    "ieee_card2",
]

df = df.dropna(
    subset=feature_columns + ["label"]
).copy()

df["label"] = df["label"].astype(int)

print("Rows after cleaning:", len(df))

# ------------------------------------------------------------
# 5. BUILD X / y
# ------------------------------------------------------------

X = df[feature_columns].astype(float).to_numpy()

y = df["label"].to_numpy()

print("Feature matrix:", X.shape)
print("Labels:", y.shape)
print("Positive fraud labels:", int(y.sum()))
print("Negative labels:", int((y == 0).sum()))

# ------------------------------------------------------------
# 6. CHRONOLOGICAL SORT
# ------------------------------------------------------------

dt_idx = 0

order = np.argsort(X[:, dt_idx])

X = X[order]
y = y[order]

# ------------------------------------------------------------
# 7. STRICT 80/20 CHRONOLOGICAL HOLDOUT
# ------------------------------------------------------------

split = int(len(X) * 0.80)

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)

print("Chronological field: ieee_dt / TransactionDT")

print(
    "Training dt:",
    X_train[0, dt_idx],
    "->",
    X_train[-1, dt_idx]
)

print(
    "Validation dt:",
    X_test[0, dt_idx],
    "->",
    X_test[-1, dt_idx]
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_test))

print("Training fraud:", int(y_train.sum()))
print("Validation fraud:", int(y_test.sum()))

# ------------------------------------------------------------
# 8. TRAIN RANDOM FOREST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# ------------------------------------------------------------
# 9. PREDICTIONS
# ------------------------------------------------------------

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 10. METRICS
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

cm = confusion_matrix(
    y_test,
    y_pred
)

# ------------------------------------------------------------
# 11. FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL_CHRONOLOGICAL_RESULTS")
print("=" * 70)

print(f"ROC_AUC={roc_auc:.6f}")
print(f"PR_AUC={pr_auc:.6f}")
print(f"ACCURACY={accuracy:.6f}")
print(f"FRAUD_PRECISION={precision:.6f}")
print(f"FRAUD_RECALL={recall:.6f}")
print(f"FRAUD_F1={f1:.6f}")

print("CONFUSION_MATRIX=")
print(cm)

print("=" * 70)
print("END_VALIDATION")
print("=" * 70)

ASSURANCE IEEE-CIS CHRONOLOGICAL VALIDATION
Dataset: /kaggle/input/datasets/lnasiri007/ieeecis-fraud-detection/train_transaction.csv
Raw dataset rows: 590540
Rows after cleaning: 581607
Feature matrix: (581607, 5)
Labels: (581607,)
Positive fraud labels: 20240
Negative labels: 561367

DATA SPLIT
Chronological field: ieee_dt / TransactionDT
Training dt: 86401.0 -> 12189601.0
Validation dt: 12189614.0 -> 15811131.0
Training rows: 465285
Validation rows: 116322
Training fraud: 16200
Validation fraud: 4040

TRAINING

FINAL_CHRONOLOGICAL_RESULTS
ROC_AUC=0.768581
PR_AUC=0.141512
ACCURACY=0.837924
FRAUD_PRECISION=0.108391
FRAUD_RECALL=0.507426
FRAUD_F1=0.178626
CONFUSION_MATRIX=
[[95419 16863]
 [ 1990  2050]]
END_VALIDATION
